# TR-01 · Witnessing an API 610 performance test (exercises)

Companion of the training guide
[TR-01](../docs/training/TR-01-witness-performance-test.md).  Every exercise of the guide
is worked here with the `pump` library, and the last cell **asserts** the answers.

The tolerance tables come from `pump/standards/api610.toml` and
`pump/standards/clients/n553.toml`; they were checked against API 610 8th, 11th, 12th
editions and Petrobras N-553 rev E (clause references in each table's `source`).

In [ ]:
from pump import Q_
from pump.acceptance import AcceptanceCriteria, deviation, head_rises_to_shutoff
from pump.npsh import check_npsh

api8 = AcceptanceCriteria.load("api610:8")
api11 = AcceptanceCriteria.load("api610:11")
b34004 = AcceptanceCriteria.load("api610:8", client="n553")
print(b34004.label, "+", b34004.client_label)
print("verified performance tolerances:", b34004.verified,
      "| open parts:", b34004.unverified)

## Exercise 1 · Which band? (rated head 120 m)

The band is chosen by the **rated** head, for the shut-off check too.  Note the shut-off
edge: 150 m in the 8th edition, 75 m in the 11th.

In [ ]:
h = Q_(120, "m")
ex1 = {
    "8th head": api8.head_tolerance(h), "8th shut-off": api8.shutoff_tolerance(h),
    "11th head": api11.head_tolerance(h), "11th shut-off": api11.shutoff_tolerance(h),
}
for k, (minus, plus) in ex1.items():
    print(f"{k:14s} -{minus:.0%} / +{plus:.0%}")

## Exercise 2 · B-34004A head (rated 333.8 m, predicted 339.5 m)

In [ ]:
rated_h, pred_h = Q_(333.8, "m"), Q_(339.5, "m")
dh = deviation(rated_h, pred_h)
minus, plus = b34004.head_tolerance(rated_h)
head_ok = -minus <= dh <= plus
print(f"dH = {dh:+.2%}  band -{minus:.0%}/+{plus:.0%}  -> {'PASS' if head_ok else 'FAIL'}"
      f"  (room left {plus - dh:.2%})")
print("under the 11th edition the band is", api11.head_tolerance(rated_h))

## Exercise 3 · Power (rated 26.9 kW, predicted 24.6 kW)

+4 % from all causes, after the corrections; a lower power is not a defect.

In [ ]:
dp = deviation(Q_(26.9, "kW"), Q_(24.6, "kW"))
power_ok = dp <= b34004.power_tol_plus
print(f"dP = {dp:+.1%}  limit +{b34004.power_tol_plus:.0%}  -> {'PASS' if power_ok else 'FAIL'}")

## Exercise 4 · Efficiency 3.7 % low, 37 kW driver

API 610 does not rate efficiency; N-553 rev E asks for -0 % only for drivers of 1 MW or
more.  The criteria carry **no** efficiency tolerance, so efficiency is information only.

In [ ]:
de = deviation(Q_(32.7, "percent"), Q_(31.5, "percent"))
print(f"d_eta = {de:+.1%}; efficiency tolerance in the criteria: {b34004.efficiency_tol_minus}")

## Exercise 5 · NPSH (NPSHa 2.4 m, NPSH3 1.6 m, quoted 1.5 m)

Two rules: the client margin (N-553 rev E: >= 0.8 m and ratio >= 1.15) and API 610's
rated NPSH +0 % (tested NPSH3 <= quoted).

In [ ]:
res = check_npsh(Q_(2.4, "m"), Q_(1.6, "m"), b34004.npsh, npshr_quoted=Q_(1.5, "m"))
print(f"margin {res.margin:.2f~P}, ratio {res.ratio:.2f}, quoted ok: {res.quoted_ok}"
      f" -> NPSH verdict {res.status.upper()}")
needs_all_points = bool(res.margin < Q_(1.0, "m") or res.ratio < 1.20)
print("N-553 rev E 9.22: NPSHr at every test point required:", needs_all_points)

## Exercise 6 · Vibration 0.16 in/s

The limit is 3.0 mm/s **RMS** (8th Table 2-5).  True peak may not be used for acceptance.

In [ ]:
v = Q_(0.16, "inch/s").to("mm/s")
limit = b34004.mechanical_run.role_max["vibration_unfiltered"]
vib_fails_if_rms = bool(v > limit)
print(f"0.16 in/s = {v:.2f~P}; limit {limit:~P} RMS -> "
      f"{'FAIL' if vib_fails_if_rms else 'PASS'} if the reading is RMS; "
      "if true peak: not usable, ask for RMS")

## Exercise 7 · Test speed (rated 10 212 rpm, tested 10 301 rpm)

In [ ]:
dn = deviation(Q_(10212, "rpm"), Q_(10301, "rpm"))
speed_ok = abs(dn) <= 0.03
print(f"dN = {dn:+.2%} -> {'within' if speed_ok else 'outside'} 3 %")

## Exercise 8 · Rising curve specified, shut-off 3 % low, curve droops

The negative shut-off tolerance is allowed only if the test curve still rises to
shut-off (`head_rises_to_shutoff`).  Two illustrative fitted curves through the same
rated point (9.8 m3/h, 333.8 m); the rated shut-off head is taken as 345 m.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def poly(c):
    return lambda q: Q_(float(np.polyval(c, q.to("m**3/h").magnitude)), "m")

rated_q, rated_so = Q_(9.8, "m**3/h"), Q_(345.0, "m")
rising = poly([-0.24, 0.0, 356.85])          # head falls steadily with flow
drooping = poly([-0.04819, 0.3855, 334.65])  # head peaks near 4 m3/h, then falls
for name, f in (("rising", rising), ("drooping", drooping)):
    so = f(Q_(0, "m**3/h"))
    print(f"{name:9s} H(0) = {so:.1f~P} ({deviation(rated_so, so):+.1%} vs 345 m), "
          f"H(rated) = {f(rated_q):.1f~P}, rises to shut-off: "
          f"{head_rises_to_shutoff(f, rated_q)}")

qs = np.linspace(0, 12, 100)
for name, f in (("rising", rising), ("drooping", drooping)):
    plt.plot(qs, [f(Q_(q, "m**3/h")).magnitude for q in qs], label=name)
plt.axvline(9.8, ls=":", c="k")
plt.xlabel("Q (m3/h)"); plt.ylabel("Head (m)"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

## Exercise 9 · The test data summary's vibration columns

`vibration_by_region` fills the summary's rated / preferred / allowable columns from the
per-point RMS readings (rated = within 5 % of rated flow; preferred = 70-120 % of BEP;
allowable = every point except shut-off, limit +30 %).

In [ ]:
from pump.utilities.test_summary import vibration_by_region

readings = [(0.0, None), (5.0, 2.8), (9.9, 2.1), (11.0, 2.6), (12.5, 3.4)]
pts = [(Q_(q, "m**3/h"), {"vib_de": Q_(v, "mm/s")}) for q, v in readings if v]
limits = {"overall": b34004.mechanical_run.role_max["vibration_unfiltered"],
          "filtered": b34004.mechanical_run.role_max["vibration_filtered"]}
(de,) = vibration_by_region(pts, Q_(10, "m**3/h"), Q_(10, "m**3/h"), limits)
print("rated", de.rated, "| preferred", de.por, "| allowable", de.aor)

## The whole verdict at once

The workbench's Compliance node and `PerformanceChecker` use one engine,
`evaluate_performance`.  Every row is `pass`, `fail`, `info` (efficiency without a
purchaser tolerance) or `not evaluated`; a verdict with a `not evaluated` row is
**NOT FULLY ASSESSED**, never an approval.  Here with the exercise numbers and a curve
that cannot predict power (no power column was recorded):

In [ ]:
from pump.acceptance import evaluate_performance

class TrainingCurve:
    points = ()
    def __init__(self, with_power=True):
        self.with_power = with_power
    def predict_head(self, q):
        return Q_(357.0 - 0.18 * q.to("m**3/h").magnitude ** 2, "m")
    def predict_breaking_power(self, q):
        if not self.with_power:
            raise AttributeError("no power column")
        return Q_(24.6, "kW")
    def predict_efficiency(self, q):
        return Q_(31.5, "percent")

def show(v):
    for r in v.rows:
        dev = f"{r.deviation:+.2%}" if r.deviation is not None else "-"
        print(f"  {r.name:12s} {r.status:14s} {dev:>8s}  {r.note}")
    print("  ->", "APPROVED" if v.approved else
          ("NOT FULLY ASSESSED" if v.passed else "NOT APPROVED"))

kw = dict(rated_power=Q_(26.9, "kW"), rated_efficiency=Q_(32.7, "percent"),
          rated_speed=Q_(10212, "rpm"), test_speeds=(Q_(10301, "rpm"),))
full = evaluate_performance(TrainingCurve(), rated_q, rated_h, b34004, **kw)
partial = evaluate_performance(TrainingCurve(False), rated_q, rated_h, b34004, **kw)
print("complete test data:"); show(full)
print("no power column:"); show(partial)

## Answers (asserted)

In [ ]:
assert de.rated == (2.1, 3.0) and de.por == (2.6, 3.0) and de.aor == (3.4, 3.9)
assert full.approved and full.row("Efficiency").status == "info"
assert not partial.approved and partial.passed and partial.not_evaluated == ("Power",)

assert ex1["8th head"] == (0.02, 0.05) and ex1["8th shut-off"] == (0.10, 0.10)
assert ex1["11th head"] == (0.03, 0.03) and ex1["11th shut-off"] == (0.08, 0.08)

assert head_ok and abs(dh - 0.0171) < 5e-4 and api11.head_tolerance(rated_h) == (0.03, 0.03)
assert power_ok and abs(dp + 0.0855) < 5e-4
assert b34004.efficiency_tol_minus is None  # efficiency: information only
assert res.status == "fail" and res.quoted_ok is False and needs_all_points
assert abs(v.magnitude - 4.064) < 1e-3 and vib_fails_if_rms
assert speed_ok and abs(dn - 0.00872) < 1e-4

assert abs(drooping(rated_q).magnitude - 333.8) < 0.01
assert head_rises_to_shutoff(rising, rated_q)
assert not head_rises_to_shutoff(drooping, rated_q)
# the drooping curve's shut-off is 3 % below 345 m: inside +/-5 %, yet it fails the rule
so_dev = deviation(rated_so, drooping(Q_(0, "m**3/h")))
assert abs(so_dev + 0.03) < 1e-3
lo, hi = api8.shutoff_tolerance(Q_(333.8, "m"))
assert -lo <= so_dev <= hi
print("TR-01 answers OK")